## Generative Adversarial Networks
The primary objective of **Generative Adversarial Network (GAN)** is to create images that resemble (but are not identical to) those in the training dataset. Including 2 neural networks that are trained in opposition to each other:
- **Generator** takes a random vector and is tasked with producing an image from it
- **Discriminator** is a network designed to differentiate between an *original image* and one created by the **generator**. 

In [ ]:
import torch
import torchvision
import matplotlib.pyplot as plt
from torchvision import transforms
from torch import nn
from torch import optim
from tqdm import tqdm
import numpy as np
import torch.nn.functional as F
torch.manual_seed(42)
np.random.seed(42)

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'

train_size = 1.0
lr = 2e-4
weight_decay = 8e-9
beta1 = 0.5
beta2 = 0.999
batch_size = 256
epochs = 100
plot_every = 10

### Generator
The generator's job is to take a random vector of a certain size *(similar to a latent vector in autoencoders)* and produce the desired image. Its function closely resembles the generative part of an autoencoder.

In [ ]:
class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.linear1 = nn.Linear(100, 256)
        self.bn1 = nn.BatchNorm1d(256, momentum=0.2)
        
        self.linear2 = nn.Linear(256, 512)
        self.bn2 = nn.BatchNorm1d(512, momentum=0.2)
        
        self.linear3 = nn.Linear(512, 1024)
        self.bn3 = nn.BatchNorm1d(1024, momentum=0.2)
        
        self.linear4 = nn.Linear(1024, 28*28)
        
        self.tanh = nn.Tanh()
        self.leaky_relu = nn.LeakyReLU(0.2)
        
    def forward(self, x):
        hidden1 = self.leaky_relu(self.bn1(self.linear1(x)))
        hidden2 = self.leaky_relu(self.bn2(self.linear2(hidden1)))
        hidden3 = self.leaky_relu(self.bn3(self.linear3(hidden2)))
        output = self.tanh(self.linear4(hidden3)).view(x.shape[0], 1, 28, 28)
        
        return output

There are a few tricks:
- Instead of `ReLU`, we use `LeakyReLU`, which is a `ReLU` that doesn't output exactly 0 for negative `x`, but instead applies another linear function with a very small slope.
- We use `BatchNorm1D` to stabilize training.
- The activation function in the last layer is `Tanh`, ensuring the output falls within the range [-1,1].

### Discriminator
Discriminator is a traditional image classification network. In our first example, we will also use a linear classifier.

In [ ]:
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.linear1 = nn.Linear(28*28, 512)
        self.linear2 = nn.Linear(512, 256)
        self.linear3 = nn.Linear(256, 1)
        
        self.leaky_relu = nn.LeakyReLU(0.2)
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, x):
        x = x.view(x.shape[0], -1)
        hidden1 = self.leaky_relu(self.linear1(x))
        hidden2 = self.leaky_relu(self.linear2(hidden1))
        output = self.sigmoid(self.linear3(hidden2))
        
        return output

### Loading the dataset

In [ ]:
def mnist(train_part, transform=None):
    dataset = torchvision.datasets.MNIST('.', download=True, transform=transform)
    train_part = int(train_part * len(dataset))
    train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_part, len(dataset) - train_part])
    return train_dataset, test_dataset

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset, test_dataset = mnist(train_size, transform=transform)
train_loader = torch.utils.data.DataLoader(train_dataset, drop_last = True, batch_size=batch_size, shuffle=True)

dataloaders = {train_loader, }

### Network training
Two phases for training:
- **Generator training**: Create random vectors called **noise**, generate **true labels** (a vector with shape (bs, 1) filled with 1.0 values), and calculate the **generator loss** (between the output of the **frozen discriminator** (with noise at input) and the **true labels**).
- **Discriminator training**: The discriminator loss is calculated in two parts: 
    - The loss between the output of the discriminator (with noise as input) and **fake labels** (a vector with shape (bs, 1) filled with 0.0 values).
    - The loss between the output of the discriminator (with real images as input) and **true labels**.
- **Final Loss** = (first_part_loss + second_part_loss) / 2

In [ ]:
def plotn(n, generator, device):
    generator.eval()
    noise = torch.FloatTensor(np.random.normal(0, 1, (n, 100))).to(device)
    imgs = generator(noise).detach().cpu()
    fig, axs = plt.subplots(1, n, figsize=(n, 1))
    for i, im in enumerate(imgs):
        axs[i].imshow(im[0])
    
    plt.show()

In [ ]:
def train_gan(dataloaders, models, optimizers, loss_fn, epochs, plot_every, decive):
    tqdm_iter = tqdm(range(epochs))
    train_loader = dataloaders[0]
    
    generator, discriminator = models
    optimizer_generator, optimizer_discriminator = optimizers
    
    for epoch in tqdm_iter:
        generator.train()
        discriminator.train()
        
        train_gen_loss = 0.0
        train_disc_loss = 0.0
        
        test_gen_loss = 0.0
        test_disc_loss = 0.0
        
        for batch in train_loader:
            real_imgs, _ = batch
            real_imgs = real_imgs.to(device)
            
            discriminator.eval()
            generator.zero_grad()
            
            noise = torch.FloatTensor(np.random.normal(0, 1, (real_imgs.shape[0], 100))).to(device)
            real_labels = torch.ones(real_imgs.shape[0], 1).to(device)
            fake_labels = torch.zeros(real_imgs.shape[0], 1).to(device)
            
            generated_imgs = generator(noise)
            discriminator_preds = discriminator(real_imgs)
            
            gen_loss = loss_fn(discriminator(generated_imgs), real_labels)
            gen_loss.backward()
            optimizer_generator.step()
            
            discriminator.train()
            discriminator.zero_grad()
            
            discriminator_real = discriminator(real_imgs)
            discriminator_real_loss = loss_fn(discriminator_real, real_labels)
            
            discriminator_fake = discriminator(generated_imgs.detach())
            discriminator_fake_loss = loss_fn(discriminator_fake, fake_labels)
            
            disc_loss = discriminator_real_loss + discriminator_fake_loss
            disc_loss.backward()
            optimizer_discriminator.step()
            
            train_gen_loss += gen_loss.item()
            train_disc_loss += disc_loss.item()
            
        train_gen_loss /= len(train_loader)
        train_disc_loss /= len(train_loader)
        
        if epoch % plot_every == 0 or epoch == epochs - 1:
            plotn(5, generator, device)
            
        tqdm_dct = {'generator loss:': train_gen_loss, 'discriminator loss:': train_disc_loss}
        tqdm_iter.set_postfix(tqdm_dct, refresh=True)
        tqdm_iter.refresh()

In [ ]:
generator = Generator().to(device)
discriminator = Discriminator().to(device)
optimizer_generator = optim.Adam(generator.parameters(), lr=lr, betas=(beta1, beta2), weight_decay=weight_decay)
optimizer_discriminator = optim.Adam(discriminator.parameters(), lr=lr, betas=(beta1, beta2), weight_decay=weight_decay)
loss_fn = nn.BCELoss()

models = (generator, discriminator)
optimizers = (optimizer_generator, optimizer_discriminator)
train_gan(dataloaders, models, optimizers, loss_fn, epochs, plot_every, device)

## Deep Convolutional GANs (DCGANs)
This model uses convolutional layers for both the generator and the discriminator. The key distinction here is the use of `Conv2DTranspose` layer in the **generator**. 

In [ ]:
class DCGenerator(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.conv1 = nn.ConvTranspose2d(100, 256, kernel_size=3, stride=2, bias = False)
        self.bn1 = nn.BatchNorm2d(256)
        
        self.conv2 = nn.ConvTranspose2d(256, 128, kernel_size=3, stride=2, bias = False)
        self.bn2 = nn.BatchNorm2d(128)
        
        self.conv3 = nn.ConvTranspose2d(128, 64, kernel_size=3, stride=2, bias = False)
        self.bn3 = nn.BatchNorm2d(64)
        
        self.conv4 = nn.ConvTranspose2d(64, 1, kernel_size=3, stride=2, padding = 2, output_padding=1, bias = False)
        self.relu = nn.ReLU()
        self.tanh = nn.Tanh()
        
    def forward(self, x):
        hidden1 = self.relu(self.bn1(self.conv1(x)))
        hidden2 = self.relu(self.bn2(self.conv2(hidden1)))
        hidden3 = self.relu(self.bn3(self.conv3(hidden2)))
        output = self.tanh(self.conv4(hidden3)).view(x.shape[0], 1, 28, 28)
        
        return output

In [ ]:
class DCDiscriminator(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.conv1 = nn.Conv2d(1, 64, kernel_size=4, stride=2, padding=1)

        self.conv2 = nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1)
        self.bn2 = nn.BatchNorm2d(128)
        
        self.conv3 = nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1)
        self.bn3 = nn.BatchNorm2d(256)
        
        self.conv4 = nn.Conv2d(256, 1, kernel_size=4, stride=2, padding=1)        
        
        self.leaky_relu = nn.LeakyReLU(0.2)
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, x):
        hidden1 = self.leaky_relu(self.conv1(x))
        hidden2 = self.leaky_relu(self.bn2(self.conv2(hidden1)))
        hidden3 = self.leaky_relu(self.bn3(self.conv3(hidden2)))
        output = self.sigmoid(self.conv4(hidden3)).view(x.shape[0], -1)
                
        return output

In [ ]:
def weight_init(model):
    classname = model.__class__.__name__
    if classname.find('Conv') != -1:
        nn.init.normal_(model.weight.data, 0.0, 0.02)
    if classname.find('BatchNorm') != -1:
        nn.init.normal_(model.weight.data, 1.0, 0.02)
        nn.init.constant_(model.bias.data, 0)

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
])

In [ ]:
train_dataset, test_dataset = mnist(train_size, transform=transform)
train_loader = torch.utils.data.DataLoader(train_dataset, drop_last = True, batch_size=batch_size, shuffle=True)
dataloaders = (train_loader, )

generator = DCGenerator().to(device)
discriminator = DCDiscriminator().to(device)
models = (generator, discriminator)
optimizer_generator = optim.Adam(generator.parameters(), lr=lr, betas=(beta1, beta2), weight_decay=weight_decay)
optimizer_discriminator = optim.Adam(discriminator.parameters(), lr=lr, betas=(beta1, beta2), weight_decay=weight_decay)
optimizers = (optimizer_generator, optimizer_discriminator)

loss_fn = nn.BCELoss()

In [ ]:
def dcplotn(n, generator, device):
    generator.eval()
    noise = torch.FloatTensor(np.random.normal(0, 1, (n, 100, 1, 1))).to(device)
    imgs = generator(noise).detach().cpu()
    fig, axs = plt.subplots(1, n)
    for i, im in enumerate(imgs):
        axs[i].imshow(im[0])
    
    plt.show()

In [ ]:
def train_dcgan(dataloaders, models, optimizers, loss_fn, epochs, plot_every, device):
    tqdm_iter = tqdm(range(epochs))
    train_loader = dataloaders[0]
    
    generator, discriminator = models
    optimizer_generator, optimizer_discriminator = optimizers
    
    for epoch in tqdm_iter:
        generator.train()
        discriminator.train()
        
        train_gen_loss = 0.0
        train_disc_loss = 0.0
        
        test_gen_loss = 0.0
        test_disc_loss = 0.0
        
        for batch in train_loader:
            real_imgs, _ = batch
            real_imgs = real_imgs.to(device)
            # Normalize images to [-1, 1]
            real_img = 2.0 * real_imgs - 1.0

            #-----Training the generator-----#
            generator.zero_grad()
            
            noise = torch.FloatTensor(np.random.normal(0, 1, (real_imgs.shape[0], 100, 1, 1))).to(device)
            real_labels = torch.ones(real_imgs.shape[0], 1).to(device)
            fake_labels = torch.zeros(real_imgs.shape[0], 1).to(device)
            
            generated_imgs = generator(noise)
            discriminator_preds = discriminator(generated_imgs)
            
            gen_loss = loss_fn(discriminator_preds, real_labels)
            
            #Compute the gradients
            gen_loss.backward()
            #Update the generator's parameters
            optimizer_generator.step()

            #-----Training the discriminator-----#
            discriminator.zero_grad()
            
            discriminator_real = discriminator(real_imgs)
            discriminator_real_loss = loss_fn(discriminator_real, real_labels)
            
            discriminator_fake = discriminator(generated_imgs.detach())
            discriminator_fake_loss = loss_fn(discriminator_fake, fake_labels)
            
            disc_loss = (discriminator_real_loss + discriminator_fake_loss) / 2.0
            disc_loss.backward()
            optimizer_discriminator.step()
            
            #-----Accumulate the losses for reporting-----#
            train_gen_loss += gen_loss.item()
            train_disc_loss += disc_loss.item()
            
        train_gen_loss /= len(train_loader)
        train_disc_loss /= len(train_loader)
        
        if epoch % plot_every == 0 or epoch == epochs - 1:
            dcplotn(5, generator, device)
            
        tqdm_dct = {'generator loss:': train_gen_loss, 'discriminator loss:': train_disc_loss}
        tqdm_iter.set_postfix(tqdm_dct, refresh=True)
        tqdm_iter.refresh()

In [ ]:
train_dcgan(dataloaders, models, optimizers, loss_fn, epochs // 2, plot_every // 2, device)

In [ ]:
generator.eval()
dcplotn(5, generator, device)